In [2]:
from datasets import load_dataset
from vllm import LLM, SamplingParams
import json
from openai import OpenAI
from langchain_openai import ChatOpenAI

ds1 = load_dataset("json", data_files="/mnt/data1tb/thangcn/datnv2/data/qa_documents/test.json", split="train")
# ds2 = load_dataset("csv", data_files="/mnt/data1tb/thangcn/datnv2/data/qa_documents/eval_predictions.csv", split="train")

INFO 06-14 15:21:55 [__init__.py:239] Automatically detected platform cuda.


2025-06-14 15:21:57,042	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
ds1

Dataset({
    features: ['question', 'context', 'answer', 'text', 'token_count'],
    num_rows: 100
})

In [4]:
df1 = ds1.to_pandas()
df1 = df1.drop(columns=['token_count'])
df1.text.iloc[0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nUse only the information to answer the question<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTại sao nướu răng của bé lại nhô lên?\n\nInformation:\n\n```\nảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo phần xương bên dưới, mà xương sẽ phát triển dựa theo răng. do hai răng cửa của bé quá hô và chìa ra phía trước nên phần xương ổ răng bao quanh chân răng cũng sẽ phát triển về phía trước. khi phần xương ổ nằm hẳn về phía trước như vậy thì tất nhiên nướu răng/lợi cũng sẽ nhô lên như trường hợp của con bạn. tốt nhất bạn nên đưa bé đi chỉnh hình răng sớm để cải thiện không chỉ về thẩm mỹ mà còn chức năng ăn nhai cũng như phát âm cho bé, bạn nhé! thân mến, alobacsi.comcổng thông tin tư vấn sức khỏe miễn phí\n```<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nDo hai răng cửa của bé quá hô và chìa ra phía trước nên phần xương ổ răng bao quanh chân 

In [5]:
df1

,question,context,answer,text
0,Tại sao nướu răng của bé lại nhô lên?,"ảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo...",Do hai răng cửa của bé quá hô và chìa ra phía ...,<|begin_of_text|><|start_header_id|>system<|en...
1,Ngưng isoniazid 6 ngày liệu có gây kháng thuốc?,"- chào em,cả 2 thuốc trên đều có thành phần ch...","Không, ngưng isoniazid 6 ngày không gây kháng ...",<|begin_of_text|><|start_header_id|>system<|en...
2,Bố mẹ nên làm gì khi bé dùng thuốc thoa trên v...,"- chào em,bây giờ, em nên ngưng cho bé dùng th...",Nên ngưng cho bé dùng thuốc và đưa bé đi khám ...,<|begin_of_text|><|start_header_id|>system<|en...
3,Mã tương đương của dịch vụ phẫu thuật nội soi ...,"Mã tương đương: 03.4068.0451, Tên dịch vụ kỹ t...",03.4068.0451,<|begin_of_text|><|start_header_id|>system<|en...
4,Bệnh tiểu máu có thể được chẩn đoán như thế nào?,"chào bạn,bạn không mô tả rõ những triệu chứng ...",Chẩn đoán tiểu máu cần có xét nghiệm tìm hồng ...,<|begin_of_text|><|start_header_id|>system<|en...
...,...,...,...,...
95,"Trước khi xét nghiệm HBSAg, tôi cần xác định l...","- chào em vân,trước khi câu hỏi của em, tôi cầ...",Tốt nhất là kiểm tra lại để chắc chắn. Nếu kết...,<|begin_of_text|><|start_header_id|>system<|en...
96,Tại sao xịt rửa mũi quá mạnh có thể gây viêm t...,khi xịt rửa mũi quá mạnh dịch mũi xâm nhập vào...,Dịch mũi xâm nhập vào tai giữa qua vòi nhĩ (eu...,<|begin_of_text|><|start_header_id|>system<|en...
97,Vi rút HPV lây lan qua đường nào?,"chào bạn, bạn cung cấp chưa có dấu hiệu nghi n...","Quan hệ tình dục qua đường âm đạo, hậu môn, đư...",<|begin_of_text|><|start_header_id|>system<|en...
98,Em không ghi rõ huyết áp của em thấp là thấp b...,"chào thu,em không ghi rõ huyết áp của em thấp ...",Theo mô tả thì vấn đề của em nằm nhiều ở nguyê...,<|begin_of_text|><|start_header_id|>system<|en...


In [6]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)
from textwrap import dedent
import torch

In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from accelerate import init_empty_weights

# Tải tokenizer
tokenizer = AutoTokenizer.from_pretrained("thang1943/Qwen2.5-7B-Instruct-final")

# Cấu hình quantization 8-bit
quantization_config = {
    "load_in_8bit": True,
    "llm_int8_threshold": 6.0,
    "llm_int8_skip_modules": None,
    "llm_int8_enable_fp32_cpu_offload": False,
    "llm_int8_has_fp16_weight": False,
}

# Tải model với quantization 8-bit
model = AutoModelForCausalLM.from_pretrained(
    "thang1943/Qwen2.5-7B-Instruct-final",
    device_map="auto",  # tự động phân bổ trên các thiết bị có sẵn
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
)

/home/duyhoang/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 4/4 [00:28<00:00,  7.24s/it]


In [8]:
def create_test_prompt(data_row):
    prompt = dedent(
        f"""
    {data_row["question"]}

    Information:

    ```
    {data_row["context"]}
    ```
    """
    )
    messages = [
        {
            "role": "system",
            "content": "Use only the information to answer the question",
        },
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [9]:
row = df1.iloc[0]
prompt = create_test_prompt(row)

In [10]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=True,
    temperature=0.7,     # > 0.0
    top_p=0.9,
    top_k=50,
    max_new_tokens=100,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)


Device set to use cuda:0


In [11]:
pipe(prompt)

[{'generated_text': 'Do hai răng cửa của bé quá hô và chìa ra phía trước nên phần xương ổ răng bao quanh chân răng cũng sẽ phát triển về phía trước. Khi phần xương ổ nằm hẳn về phía trước như vậy thì tất nhiên nướu răng/lợi cũng sẽ nhô lên như trường hợp của con bạn.'}]

In [12]:
from tqdm import tqdm

In [13]:
predictions = []
for index, row in tqdm(df1.iterrows()):
    outputs = pipe(create_test_prompt(row))
    predictions.append(outputs[0]["generated_text"])

9it [00:20,  2.01s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
100it [03:20,  2.00s/it]


In [14]:
df1['response'] = predictions

In [15]:
df1['response'].iloc[1]

'Không, ngưng isoniazid 6 ngày không gây kháng thuốc.'

In [16]:
df1['reference'] = df1['answer']
df1.drop(columns='answer')

,question,context,text,response,reference
0,Tại sao nướu răng của bé lại nhô lên?,"ảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo...",<|begin_of_text|><|start_header_id|>system<|en...,Do hai răng cửa của bé quá hô và chìa ra phía ...,Do hai răng cửa của bé quá hô và chìa ra phía ...
1,Ngưng isoniazid 6 ngày liệu có gây kháng thuốc?,"- chào em,cả 2 thuốc trên đều có thành phần ch...",<|begin_of_text|><|start_header_id|>system<|en...,"Không, ngưng isoniazid 6 ngày không gây kháng ...","Không, ngưng isoniazid 6 ngày không gây kháng ..."
2,Bố mẹ nên làm gì khi bé dùng thuốc thoa trên v...,"- chào em,bây giờ, em nên ngưng cho bé dùng th...",<|begin_of_text|><|start_header_id|>system<|en...,Ngừng cho bé dùng thuốc và thoa loại thuốc trê...,Nên ngưng cho bé dùng thuốc và đưa bé đi khám ...
3,Mã tương đương của dịch vụ phẫu thuật nội soi ...,"Mã tương đương: 03.4068.0451, Tên dịch vụ kỹ t...",<|begin_of_text|><|start_header_id|>system<|en...,03.4068.0451,03.4068.0451
4,Bệnh tiểu máu có thể được chẩn đoán như thế nào?,"chào bạn,bạn không mô tả rõ những triệu chứng ...",<|begin_of_text|><|start_header_id|>system<|en...,Chẩn đoán tiểu máu cần có xét nghiệm tìm hồng ...,Chẩn đoán tiểu máu cần có xét nghiệm tìm hồng ...
...,...,...,...,...,...
95,"Trước khi xét nghiệm HBSAg, tôi cần xác định l...","- chào em vân,trước khi câu hỏi của em, tôi cầ...",<|begin_of_text|><|start_header_id|>system<|en...,"Có, trước khi xét nghiệm HBSAg, cần xác định l...",Tốt nhất là kiểm tra lại để chắc chắn. Nếu kết...
96,Tại sao xịt rửa mũi quá mạnh có thể gây viêm t...,khi xịt rửa mũi quá mạnh dịch mũi xâm nhập vào...,<|begin_of_text|><|start_header_id|>system<|en...,"Khi xịt rửa mũi quá mạnh, dịch mũi xâm nhập và...",Dịch mũi xâm nhập vào tai giữa qua vòi nhĩ (eu...
97,Vi rút HPV lây lan qua đường nào?,"chào bạn, bạn cung cấp chưa có dấu hiệu nghi n...",<|begin_of_text|><|start_header_id|>system<|en...,"Quan hệ tình dục qua đường âm đạo, hậu môn, đư...","Quan hệ tình dục qua đường âm đạo, hậu môn, đư..."
98,Em không ghi rõ huyết áp của em thấp là thấp b...,"chào thu,em không ghi rõ huyết áp của em thấp ...",<|begin_of_text|><|start_header_id|>system<|en...,Theo mô tả thì vấn đề của em nằm nhiều ở nguyê...,Theo mô tả thì vấn đề của em nằm nhiều ở nguyê...


In [17]:
import os
from dotenv import load_dotenv

In [18]:
load_dotenv('/mnt/data1tb/thangcn/datnv2/.env')
open_ai_key = os.getenv("OPENAI_API_KEY")
# groq_api_key = os.getenv("GROQ_API_KEY")
MODEL = 'gpt-4o' #os.getenv("MODEL", "gpt-4o")

llm = ChatOpenAI(model=MODEL, temperature=0.3, api_key=open_ai_key)

In [19]:
df1['retrieved_contexts'] = df1['context']

In [20]:
for i in range(len(df1['retrieved_contexts'])):
    df1['retrieved_contexts'].iloc[i] = [df1['retrieved_contexts'].iloc[i]]
    

/tmp/ipykernel_225205/3607375772.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df1['retrieved_contexts'].iloc[i] = [df1['retrieved_contexts'].iloc[i]]
/tmp/ipykernel_225205/3607375772.py:2: FutureWarning: ChainedAssignmentError: behavi

In [21]:
df1['retrieved_contexts'].iloc[0]

['ảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo phần xương bên dưới, mà xương sẽ phát triển dựa theo răng. do hai răng cửa của bé quá hô và chìa ra phía trước nên phần xương ổ răng bao quanh chân răng cũng sẽ phát triển về phía trước. khi phần xương ổ nằm hẳn về phía trước như vậy thì tất nhiên nướu răng/lợi cũng sẽ nhô lên như trường hợp của con bạn. tốt nhất bạn nên đưa bé đi chỉnh hình răng sớm để cải thiện không chỉ về thẩm mỹ mà còn chức năng ăn nhai cũng như phát âm cho bé, bạn nhé! thân mến, alobacsi.comcổng thông tin tư vấn sức khỏe miễn phí']

In [22]:
from datasets import Dataset

ds = Dataset.from_pandas(df1)

In [23]:
ds

Dataset({
    features: ['question', 'context', 'answer', 'text', 'response', 'reference', 'retrieved_contexts'],
    num_rows: 100
})

In [24]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [25]:
from ragas.metrics import answer_correctness, faithfulness, answer_relevancy
from ragas import evaluate

In [26]:
EMBED_MODEL = "thang1943/multilingual-e5-large-v2" #os.getenv("EMBED_MODEL", "nampham1106/bkcare-embedding")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cpu'}
)


/tmp/ipykernel_225205/400226938.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [27]:
ragas_results = evaluate(ds, metrics=[answer_correctness, answer_relevancy, faithfulness], llm = llm, embeddings=embeddings)

Evaluating: 100%|██████████| 300/300 [01:37<00:00,  3.08it/s]


In [28]:
ragas_results

{'answer_correctness': 0.8349, 'answer_relevancy': 0.2935, 'faithfulness': 0.9452}

In [62]:
import tiktoken 

In [66]:
def count_tokens(text, model="gpt-4o"):
    encoder = tiktoken.encoding_for_model(model)
    return len(encoder.encode(text))

# --- Tính token trước khi chạy đánh giá ---
def estimate_ragas_usage(dataset):
    total_tokens = 0
    prompt_template = """
    Evaluate if the answer directly addresses the question (1/0):
    Question: {question}
    Answer: {answer}
    """
    prompt_tokens = count_tokens(prompt_template)

    for q, c, r1, r2 in zip(dataset["question"], dataset["context"], dataset["response"], dataset["reference"]):
        question_tokens = count_tokens(q)
        context_tokens = count_tokens(c)
        response_tokens = count_tokens(r1)
        reference_tokens = count_tokens(r2)

        total_tokens += (prompt_tokens + question_tokens + context_tokens + response_tokens + reference_tokens)
    
    return total_tokens

# --- Ước lượng token ---
estimated_tokens = estimate_ragas_usage(ds)
print(f"Estimated tokens needed: {estimated_tokens}")

Estimated tokens needed: 30321
